In [1]:
from pathlib import Path
import io
import zipfile

import pandas as pd
import requests

BASE_URL = ("https://reports-public.ieso.ca/public/HourlyConsumptionByFSA/")

DATA_DIR = Path("data/electricity_consumption")


In [2]:
# ---------------------------------------------------------
# Public holidays
# ---------------------------------------------------------

def get_ontario_holidays(start_year, end_year):
    """
    Return Ontario public holiday dates for the requested
    range of years.
    """

    holidays = []

    for year in range(start_year, end_year + 1):

        # -------------------------------------------------
        # New Year's Day
        # -------------------------------------------------

        holidays.append({
            "DATE": pd.Timestamp(f"{year}-01-01"),
            "HOLIDAY": "New Year's Day"
        })

        # -------------------------------------------------
        # Family Day
        # Third Monday in February
        # -------------------------------------------------

        february_1 = pd.Timestamp(f"{year}-02-01")

        family_day = february_1 + pd.Timedelta(
            (7 - february_1.weekday()) % 7 + 14,
            unit="D"
        )

        holidays.append({
            "DATE": family_day,
            "HOLIDAY": "Family Day"
        })

        # -------------------------------------------------
        # Good Friday
        # Friday before Easter
        # -------------------------------------------------

        easter = (
            pd.Timestamp(f"{year}-03-21")
            + pd.offsets.Easter()
        )

        good_friday = easter - pd.Timedelta(
            2,
            unit="D"
        )

        holidays.append({
            "DATE": good_friday,
            "HOLIDAY": "Good Friday"
        })

        # -------------------------------------------------
        # Victoria Day
        # Monday preceding May 25
        # -------------------------------------------------

        may_24 = pd.Timestamp(f"{year}-05-24")

        victoria_day = may_24 - pd.Timedelta(
            may_24.weekday(),
            unit="D"
        )

        holidays.append({
            "DATE": victoria_day,
            "HOLIDAY": "Victoria Day"
        })

        # -------------------------------------------------
        # Canada Day
        # -------------------------------------------------

        holidays.append({
            "DATE": pd.Timestamp(f"{year}-07-01"),
            "HOLIDAY": "Canada Day"
        })

        # -------------------------------------------------
        # Labour Day
        # First Monday in September
        # -------------------------------------------------

        september_1 = pd.Timestamp(f"{year}-09-01")

        labour_day = september_1 + pd.Timedelta(
            (7 - september_1.weekday()) % 7,
            unit="D"
        )

        holidays.append({
            "DATE": labour_day,
            "HOLIDAY": "Labour Day"
        })

        # -------------------------------------------------
        # Thanksgiving Day
        # Second Monday in October
        # -------------------------------------------------

        october_1 = pd.Timestamp(f"{year}-10-01")

        thanksgiving = october_1 + pd.Timedelta(
            (7 - october_1.weekday()) % 7 + 7,
            unit="D"
        )

        holidays.append({
            "DATE": thanksgiving,
            "HOLIDAY": "Thanksgiving Day"
        })

        # -------------------------------------------------
        # Christmas Day
        # -------------------------------------------------

        holidays.append({
            "DATE": pd.Timestamp(f"{year}-12-25"),
            "HOLIDAY": "Christmas Day"
        })

        # -------------------------------------------------
        # Boxing Day
        # -------------------------------------------------

        holidays.append({
            "DATE": pd.Timestamp(f"{year}-12-26"),
            "HOLIDAY": "Boxing Day"
        })

    return pd.DataFrame(holidays)
    

In [3]:
# ---------------------------------------------------------
# Generate months
# ---------------------------------------------------------

def generate_months(start_month, end_month):
    """
    Generate YYYY-MM values between start_month and end_month,
    inclusive.
    """

    start = pd.Period(start_month, freq="M")
    end = pd.Period(end_month, freq="M")

    if start > end:
        raise ValueError("start_month must be <= end_month")

    return [
        str(month)
        for month in pd.period_range(start, end, freq="M")
    ]


In [4]:
# ---------------------------------------------------------
# Download + extract one month
# ---------------------------------------------------------

def download_month(month):
    """
    Download one IESO monthly ZIP and return its contents
    as a pandas DataFrame.
    """

    month_code = month.replace("-", "")

    url = (
        BASE_URL
        + f"PUB_HourlyConsumptionByFSA_{month_code}_v1.zip"
    )

    print(f"Downloading {month}...")

    response = requests.get(
        url,
        timeout=(10, 120)
    )

    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:

        files = [
            name
            for name in z.namelist()
            if not name.endswith("/")
        ]

        if not files:
            raise ValueError(
                f"No files found inside ZIP for {month}"
            )

        # Usually the ZIP contains the data file.
        data_file = files[0]

        with z.open(data_file) as f:
            df = pd.read_csv(f, skiprows = 3)

    print(f"  Downloaded {len(df):,} rows")

    return df



In [5]:
# ---------------------------------------------------------
# Process one month
# ---------------------------------------------------------

def process_month(df, holidays):
    """
    Process one month's IESO data.
    """

    # -----------------------------------------------------
    # Dates
    # -----------------------------------------------------

    df["DATE"] = pd.to_datetime(
        df["DATE"],
        errors="coerce"
    )

    if df["DATE"].isna().any():
        raise ValueError("Some DATE values could not be parsed.")

    df["DAY_OF_WEEK"] = df["DATE"].dt.day_name()

    df["MONTH"] = df["DATE"].dt.month

    df["DAY_OF_YEAR"] = df["DATE"].dt.dayofyear

    df["IS_WEEKEND"] = (
        df["DATE"].dt.dayofweek >= 5
    ).astype("int8")

    # -----------------------------------------------------
    # Consumption per unit
    # -----------------------------------------------------

    df["CONSUMPTION_PER_UNIT"] = (
        df["TOTAL_CONSUMPTION"]
        / df["PREMISE_COUNT"]
    )

    # Avoid infinity if premise count is zero
    df["CONSUMPTION_PER_UNIT"] = (
        df["CONSUMPTION_PER_UNIT"]
        .replace([float("inf"), -float("inf")], pd.NA)
    )

    # -----------------------------------------------------
    # Public holiday
    # -----------------------------------------------------

    holiday_dates = set(holidays["DATE"])

    df["IS_PUBLIC_HOLIDAY"] = (
        df["DATE"]
        .isin(holiday_dates)
        .astype("int8")
    )

    # -----------------------------------------------------
    # Remove broader geographic classifications
    # -----------------------------------------------------

    df["FSA"] = (
        df["FSA"]
        .astype("string")
        .str.strip()
    )

    df = df[
        df["FSA"].str.len() == 3
    ].copy()

    return df
    


In [6]:
# ---------------------------------------------------------
# Save Parquet
# ---------------------------------------------------------

def save_month(df, month):
    """
    Save one processed month as Parquet.
    """

    DATA_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = DATA_DIR / f"{month}.parquet"

    df.to_parquet(
        output_file,
        index=False
    )

    print(f"  Saved → {output_file}")


In [7]:
# ---------------------------------------------------------
# Main ETL function
# ---------------------------------------------------------

def build_dataset(start_month, end_month):
    """
    Download, process and save IESO hourly consumption data.

    Parameters
    ----------
    start_month : str
        YYYY-MM

    end_month : str
        YYYY-MM
    """

    months = generate_months(
        start_month,
        end_month
    )

    years = range(
        pd.Period(start_month).year,
        pd.Period(end_month).year + 1
    )

    holidays = get_ontario_holidays(
        min(years),
        max(years)
    )

    print(
        f"Processing {len(months)} month(s): "
        f"{start_month} → {end_month}"
    )

    for month in months:

        try:

            df = download_month(month)

            df = process_month(df,holidays)

            save_month(df,month)

            del df

        except requests.HTTPError as e:

            print(f"  ERROR downloading {month}: {e}")

        except Exception as e:

            print(f"  ERROR processing {month}: {e}")

    print("\nFinished.")


In [11]:
# ---------------------------------------------------------
# Example
# ---------------------------------------------------------

if __name__ == "__main__":

    build_dataset(
        start_month="2026-01",
        end_month="2026-01"
    )

Processing 1 month(s): 2026-01 → 2026-01
  Downloaded 1,471,618 rows
  Saved → data/electricity_consumption/2026-01.parquet

Finished.
